In [9]:
import torch
import os
from scripts.baseline.baseline_tester import BaselineTester
from model_utils.models.learning.siamese import SiameseModelPairs

In [8]:
os.getcwd()

'c:\\Users\\hbori\\Documents\\Itau_env\\VA-TE'

In [10]:


device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Build the SigLIP backbone exactly as training
tester = BaselineTester(model_type="siglip", batch_size=1, device=device)
backbone = tester.model_wrapper  # this exposes encode_text()

# 2. Build the Siamese model
model = SiameseModelPairs(
    embedding_dim=768,
    projection_dim=768,
    backbone=backbone,
).to(device)

# 3. Load your projector weights (placed in model_utils)
state = torch.load("model_utils/best_model_siglip_pair.pt", map_location=device)
model.load_state_dict(state)

model.eval()


USING STANDARD EMBEDDING EXTRACTOR


SiameseModelPairs(
  (projector): Sequential(
    (0): Linear(in_features=768, out_features=768, bias=True)
    (1): ReLU()
    (2): Linear(in_features=768, out_features=768, bias=True)
  )
)

In [13]:
emb = model.encode(["amazon", "amaz0n"])
print(emb.shape)  # (2, 768)

# cosine similarity
cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
print(cos(emb[0:1], emb[1:2]))  # tensor([0.9234])

torch.Size([2, 768])
tensor([0.9379], grad_fn=<SumBackward1>)


In [17]:
import pandas as pd
def load_pairs(path: str):
    df = pd.read_parquet(path)
    pairs = list(zip(df['fraudulent_name'].tolist(), df['real_name'].tolist()))
    labels = df['label'].tolist()
    return pairs, labels

pairs, labels = load_pairs("data/processed/validate_pairs_ref_10k.parquet")

In [18]:
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score
import numpy as np
import torch

# prepare lists
frauds = [p[0] for p in pairs]
reals = [p[1] for p in pairs]
n = len(frauds)
batch_size = 512

sims_list = []
model.eval()
with torch.no_grad():
    for i in range(0, n, batch_size):
        f_batch = frauds[i : i + batch_size]
        r_batch = reals[i : i + batch_size]

        emb_f = model.encode(f_batch).to(device)  # (b, dim)
        emb_r = model.encode(r_batch).to(device)  # (b, dim)

        s = cos(emb_f, emb_r)  # (b,)
        sims_list.append(s.cpu())

sims = torch.cat(sims_list).numpy()
y_true = np.array(labels).astype(int)

# AUC-ROC
auc = roc_auc_score(y_true, sims)

# ROC curve -> find best Youden (tpr - fpr)
fpr, tpr, thresholds = roc_curve(y_true, sims)
youden = tpr - fpr
best_idx = np.argmax(youden)
best_threshold = thresholds[best_idx]
best_youden = youden[best_idx]

# Accuracy at chosen threshold
y_pred = (sims >= best_threshold).astype(int)
acc = accuracy_score(y_true, y_pred)

print(f"auc_roc: {auc:.6f}")
print(f"best_youden: {best_youden:.6f}")
print(f"chosen_threshold: {best_threshold:.6f}")
print(f"accuracy: {acc:.6f}")

auc_roc: 0.968335
best_youden: 0.813937
chosen_threshold: 0.938676
accuracy: 0.903190


In [19]:
def save_pair_embeddings(pairs, model, batch_size=batch_size, device=device, out_dir="embeddings", filename="pairs_embeddings_validate.npz"):
    """
    Generate embeddings for all unique words in `pairs` and save to an .npz file.

    Expects `pairs`, `model`, `batch_size`, `device`, `os`, `torch`, and `np` to be available
    in the notebook (they are already defined in previous cells).
    """
    os.makedirs(out_dir, exist_ok=True)

    # preserve original order while removing duplicates
    seen = set()
    unique_names = []
    for a, b in pairs:
        if a not in seen:
            unique_names.append(a); seen.add(a)
        if b not in seen:
            unique_names.append(b); seen.add(b)

    model.eval()
    emb_chunks = []
    with torch.no_grad():
        for i in range(0, len(unique_names), batch_size):
            batch = unique_names[i : i + batch_size]
            emb = model.encode(batch)  # (b, dim) torch.Tensor
            emb = emb.detach().cpu().numpy()
            emb_chunks.append(emb)

    embeddings = np.vstack(emb_chunks) if emb_chunks else np.zeros((0, 0), dtype=np.float32)
    out_path = os.path.join(out_dir, filename)
    np.savez_compressed(out_path, names=np.array(unique_names, dtype=object), embeddings=embeddings)
    print(f"Saved {embeddings.shape[0]} embeddings (dim={embeddings.shape[1] if embeddings.size else 0}) to {out_path}")

# Example call (will run with existing `pairs` and `model` variables)
pairs, _ = load_pairs("data/processed/validate_pairs_ref_10k.parquet")
save_pair_embeddings(pairs, model)

Saved 9952 embeddings (dim=768) to embeddings\pairs_embeddings_validate.npz
